# Module 11: Interactive Network Architecture — Sockets & HTTP

### What You Will Discover
By running this notebook, you will explore raw TCP sockets, solve the packet framing boundary dilemma, inspect HTTP/1.1 wire protocol bytes, and make resilient async HTTP requests with `httpx`.

**Key Question Answered:** *Why does `socket.recv(1024)` not guarantee receiving the complete message sent by `socket.send(data)`?*


In [ ]:
# Step 1: Raw socket address resolution and loopback setup
import socket

hostname = 'localhost'
ip_addr = socket.gethostbyname(hostname)
print(f'Resolved {hostname} -> {ip_addr}')


In [ ]:
# Step 2: Binary message packaging with length prefix (struct)
import struct

payload = b'HealthCheck: STATUS_OK'
# 4-byte big-endian unsigned int header + payload
packet = struct.pack('!I', len(payload)) + payload
print(f'Encoded packet: {packet[:8]}... (Total length: {len(packet)} bytes)')


In [ ]:
# Step 3: Unpacking and verifying length header
header = packet[:4]
(msg_len,) = struct.unpack('!I', header)
body = packet[4:4+msg_len]
print(f'Decoded length header: {msg_len} bytes, Body: {body.decode("utf-8")}')


### 🔮 Prediction Prompt
**Before running the next cell:** In an HTTP/1.1 GET request sent over a raw TCP socket, what exact sequence of 4 characters separates the HTTP headers from the request body? Write it down.


In [ ]:
# Surprising Result: HTTP/1.1 Header Delimiter (\r\n\r\n)
raw_http_request = (
    b'GET /health HTTP/1.1\r\n'
    b'Host: localhost:8080\r\n'
    b'User-Agent: DiagnosticClient/1.0\r\n'
    b'\r\n'  # The empty line separating headers from body
)
headers, _, body = raw_http_request.partition(b'\r\n\r\n')
print(f'Headers size: {len(headers)} bytes')
print('Explanation: HTTP/1.1 uses \r\n\r\n (CRLF CRLF) to mark the end of headers!')


### Resilient Async HTTP with `httpx`
Modern Python services use `httpx.AsyncClient` for persistent connection pooling and timeout handling.


In [ ]:
import httpx

# Simulating client initialization with connection limits and timeouts
limits = httpx.Limits(max_keepalive_connections=5, max_connections=10)
timeout = httpx.Timeout(5.0, connect=2.0)
client = httpx.AsyncClient(limits=limits, timeout=timeout)
print('Initialized httpx.AsyncClient with keep-alive pooling.')
await client.aclose()


### Socket Reuse with `SO_REUSEADDR`
Prevents `OSError: [Errno 98] Address already in use` when quickly restarting servers.


In [ ]:
test_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
test_sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
print('Successfully set SO_REUSEADDR socket option.')
test_sock.close()


### 🛠️ Interactive Challenge: Fix Incomplete Socket Reads
The following function attempts to read an exact number of bytes from a socket, but calls `recv()` only once, which fails under network fragmentation. Fix it to loop until all expected bytes are received.


In [ ]:
# TODO: FIX ME - Implement a loop to guarantee reading exact N bytes
class MockSocket:
    def __init__(self, data: bytes):
        self.data = data
        self.idx = 0
    def recv(self, bufsize: int):
        # Simulates network packet fragmentation (returns 2 bytes at a time)
        chunk = self.data[self.idx:self.idx + 2]
        self.idx += len(chunk)
        return chunk

def read_exact(sock, length: int) -> bytes:
    buffer = b''
    # FIX: while len(buffer) < length: chunk = sock.recv(length - len(buffer)); buffer += chunk
    while len(buffer) < length:
        chunk = sock.recv(length - len(buffer))
        if not chunk:
            break
        buffer += chunk
    return buffer

sock = MockSocket(b'SECRET_TOKEN_PAYLOAD')
data = read_exact(sock, 12)
print(f'Successfully read exact 12 bytes across fragments: {data}')


### 🏁 Summary & Next Steps
- TCP is a continuous byte stream; application framing is mandatory.
- Use length prefixes or `\r\n\r\n` delimiters to demarcate messages.
- Always use `sendall()` and loop on `recv()`.
- Run `python 01_tcp_socket_echo_demo.py` and `python 02_http_parsing_and_httpx_demo.py`.
- Follow [PROJECT_GUIDE.md](PROJECT_GUIDE.md) to implement the HTTP reverse proxy.
